# Chapter 39
## Short-Term Depression and Facilitation
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter39.ipynb)

## About this chapter

Synapses can change strength through a pulse train even with fixed anatomy.
The examples below first introduce a smooth spike/pulse approximation
motivating the synaptic gating variable, then show depressing and
facilitating synapses in RTM and WB neurons.

Depression depletes available resources after release and recovers between
events. Facilitation transiently increases utilization, so a pulse sequence
can initially grow. Their balance changes effective synaptic conductance,
postsynaptic timing, and whether early or late pulses evoke spikes.

With resources $x$ and utilization $u$, release is proportional to $ux$;
depression lowers $x$ after events and facilitation raises $u$. The
resulting synaptic gate enters the conductance current $gs(E_{\rm
rev}-v)$. The introductory smooth proxy is $\gamma=1+\tanh(v/10)$, which
approximates a spike-shaped pulse for a voltage trace.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
from types import SimpleNamespace

import numpy as np
from numpy import exp, tanh
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact

## Pulses

`PULSES` plots an RTM voltage trace together with its smooth
$\gamma=1+\tanh(v/10)$ spike/pulse approximation, including the per-spike
area under $\gamma$. This is not a resource/utilization simulation -- it
just motivates why $\gamma$ can stand in for a delta-function-like spike
when driving a synaptic gate. Compare the voltage crossing with the smooth
$\gamma$ pulse.

In [ ]:
# ------------------------------------------------------- RTM gating functions
# shared by PULSES, RTM_WITH_DEPRESSING_AND_FACILITATING_S, and
# RTM_WITH_DEPRESSING_S below

g_k, g_na, g_l = 80., 100., 0.1
v_k, v_na, v_l = -100., 50., -67.


def alpha_h(v):
    return 0.128 * exp(-(v + 50.0) / 18.0)


def alpha_m(v):
    return 0.32 * (v + 54) / (1.0 - exp(-(v + 54.0) / 4.0))


def alpha_n(v):
    return 0.032 * (v + 52) / (1.0 - exp(-(v + 52.0) / 5.0))


def beta_h(v):
    return 4.0 / (1.0 + exp(-(v + 27.0) / 5.0))


def beta_m(v):
    return 0.28 * (v + 27.0) / (exp((v + 27.0) / 5.0) - 1.0)


def beta_n(v):
    return 0.5 * exp(-(v + 57.0) / 40.0)


def h_inf(v):
    return alpha_h(v) / (alpha_h(v) + beta_h(v))


def m_inf(v):
    return alpha_m(v) / (alpha_m(v) + beta_m(v))


def n_inf(v):
    return alpha_n(v) / (alpha_n(v) + beta_n(v))

In [ ]:
def derivative_pulses(x0, t, i_ext):
    '''Traub (RTM) model, no synapse.'''
    v, m, n, h = x0
    dv = i_ext - g_na * h * m ** 3 * \
        (v - v_na) - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l)
    dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
    dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
    dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
    return [dv, dm, dn, dh]


def simulate_pulses(i_ext=0.5, t_final=22., dt=0.01, C=1.0):
    v0 = -70.0
    x0 = [v0, m_inf(v0), n_inf(v0), h_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative_pulses, x0, t, args=(i_ext,))
    v = sol[:, 0]

    num_spikes = 0
    for i in range(len(v) - 1):
        if v[i] < -20 and v[i + 1] >= -20:
            num_spikes += 1
    gamma = C * (1 + np.tanh(v / 10.0))
    integral = gamma.sum() * dt / num_spikes if num_spikes else np.nan
    return SimpleNamespace(t=t, v=v, gamma=gamma, num_spikes=num_spikes, integral=integral)


def plot_pulses(result):
    fig, ax = plt.subplots(2, figsize=(5, 4), sharex=True)
    ax[0].plot(result.t, result.v, lw=2, c="k")
    ax[1].plot(result.t, result.gamma, lw=2, c='k')
    ax[0].set_ylabel("v [mV]")
    ax[1].set_ylabel('1+tanh(v/10)')
    ax[1].set_xlabel("time [ms]")
    ax[0].set_yticks(range(-100, 100, 50))
    ax[0].set_xlim(18, 22)
    ax[0].set_ylim(-100, 50)
    plt.tight_layout()
    return fig

In [ ]:
def _pulses_widget(i_ext=0.5):
    result = simulate_pulses(i_ext=i_ext)
    print(f"integral per period: {result.integral:.4f}  "
          f"(1/integral = {1.0 / result.integral:.4f})")
    plot_pulses(result)


interact(_pulses_widget, i_ext=(0.2, 1.0, 0.05));

## RTM with depressing and facilitating synapse

`RTM_WITH_DEPRESSING_AND_FACILITATING_S` compares both mechanisms acting
together on an RTM cell's own synapse: depression lowers the resource
fraction $p$ after each release while facilitation raises the per-spike
utilization $U=1-e^{-W}$. Track $p$, $q$, $s$, and $U$ over the pulse
train -- depression should weaken later events until recovery, while
facilitation can enhance them transiently.

In [ ]:
def derivative_rtm_depressing_facilitating(x0, t, i_ext, C, tau_rec, tau_d_q,
                                            tau_r, tau_d, tau_facil, W_0, w):
    v, m, n, h, p, q, s, W = x0
    dv = i_ext - g_na * h * m ** 3 * \
        (v - v_na) - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l)
    dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
    dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
    dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
    dp = -C * (1 + np.tanh(0.1 * v)) * p * W + (1 - p - q) / tau_rec
    dq = C * (1 + np.tanh(0.1 * v)) * p * W - q / tau_d_q
    ds = q * (1 - s) / tau_r - s / tau_d
    dW = -(np.exp(W - W_0) - 1) / tau_facil + C * (1 + np.tanh(0.1 * v)) * w

    return [dv, dm, dn, dh, dp, dq, ds, dW]


def simulate_rtm_depressing_facilitating(i_ext=0.5, t_final=400., dt=0.01,
                                          tau_facil=500., tau_rec=300.,
                                          tau_d_q=5., tau_r=3., tau_d=9.,
                                          C=1.545, U_0=0.1, u=0.2):
    W_0 = np.log(1 / (1 - U_0))
    w = np.log(1 / (1 - u))
    v0 = -70.0
    x0 = [v0, m_inf(v0), n_inf(v0), h_inf(v0), 1.0, 0.0, 0.0, W_0]
    t = np.arange(0, t_final + dt, dt)
    sol = odeint(derivative_rtm_depressing_facilitating, x0, t,
                 args=(i_ext, C, tau_rec, tau_d_q, tau_r, tau_d, tau_facil, W_0, w))
    v, p, q, s, W = sol[:, 0], sol[:, 4], sol[:, 5], sol[:, 6], sol[:, 7]

    num_spikes = 0
    for i in range(len(v) - 1):
        if v[i] < -20 and v[i + 1] >= -20:
            num_spikes += 1
    gamma = C * (1 + np.tanh(v / 10.0))
    integral = gamma.sum() * dt / num_spikes if num_spikes else np.nan
    return SimpleNamespace(t=t, v=v, p=p, q=q, s=s, W=W, U=1 - np.exp(-W),
                            num_spikes=num_spikes, integral=integral)


def plot_rtm_depressing_facilitating(result):
    t, v, p, q, s, W = result.t, result.v, result.p, result.q, result.s, result.W
    fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(9, 5), sharex=True)
    ax[0, 0].plot(t, v, lw=2, c="k")
    ax[0, 1].plot(t, p, lw=2, c='k')
    ax[1, 0].plot(t, q, lw=2, c='k')
    ax[1, 1].plot(t, s, lw=2, c='k')
    ax[1, 2].plot(t, 1 - np.exp(-W), lw=2, c='k')

    ax[0, 0].set_ylabel("v [mV]")
    ax[0, 1].set_ylabel('p')
    ax[1, 0].set_ylabel('q')
    ax[1, 1].set_ylabel('s')
    ax[1, 2].set_ylabel("U")
    ax[1, 0].set_xlabel("time [ms]")
    ax[1, 1].set_xlabel("time [ms]")
    ax[1, 2].set_xlabel("time [ms]")
    ax[0, -1].axis('off')

    ax[0, 0].set_yticks(range(-100, 100, 50))
    ax[0, 0].set_ylim(-100, 50)
    ax[0, 1].set_xlim(min(t), max(t))
    ax[0, 1].set_ylim(0, 1.01)

    plt.tight_layout()
    return fig

In [ ]:
def _rtm_depressing_facilitating_widget(tau_facil=500.):
    result = simulate_rtm_depressing_facilitating(tau_facil=tau_facil)
    plot_rtm_depressing_facilitating(result)


interact(_rtm_depressing_facilitating_widget, tau_facil=(100., 1000., 50.));

## RTM with depressing synapse

`RTM_WITH_DEPRESSING_S` isolates depression alone in an RTM cell: no
facilitation variable, just the resource fraction $p$, the transiently
open gate $q$, and the resulting synaptic gate $s$.

In [ ]:
def derivative_rtm_depressing(x0, t, i_ext, C, U, tau_rec, tau_d_q, tau_r, tau_d):
    v, m, n, h, p, q, s = x0
    dv = i_ext - g_na * h * m ** 3 * \
        (v - v_na) - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l)
    dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
    dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
    dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
    dp = -C * (1 + np.tanh(0.1 * v)) * p * np.log(1 / (1 - U)) + (1 - p - q) / tau_rec
    dq = C * (1 + np.tanh(0.1 * v)) * p * np.log(1 / (1 - U)) - q / tau_d_q
    ds = q * (1 - s) / tau_r - s / tau_d

    return [dv, dm, dn, dh, dp, dq, ds]


def simulate_rtm_depressing(i_ext=0.5, t_final=200., dt=0.01,
                             U=0.5, C=1.545, tau_rec=500., tau_d_q=5.,
                             tau_r=3., tau_d=9.):
    v0 = -70.0
    x0 = [v0, m_inf(v0), n_inf(v0), h_inf(v0), 1.0, 0.0, 0.0]
    t = np.arange(0, t_final + dt, dt)
    sol = odeint(derivative_rtm_depressing, x0, t,
                 args=(i_ext, C, U, tau_rec, tau_d_q, tau_r, tau_d))
    v, p, q, s = sol[:, 0], sol[:, 4], sol[:, 5], sol[:, 6]

    num_spikes = 0
    for i in range(len(v) - 1):
        if v[i] < -20 and v[i + 1] >= -20:
            num_spikes += 1
    gamma = C * (1 + np.tanh(v / 10.0))
    integral = gamma.sum() * dt / num_spikes if num_spikes else np.nan
    return SimpleNamespace(t=t, v=v, p=p, q=q, s=s, num_spikes=num_spikes, integral=integral)


def plot_rtm_depressing(result):
    t, v, p, q, s = result.t, result.v, result.p, result.q, result.s
    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(6, 5), sharex=True)
    ax[0, 0].plot(t, v, lw=2, c="k")
    ax[0, 1].plot(t, p, lw=2, c='k')
    ax[1, 0].plot(t, q, lw=2, c='k')
    ax[1, 1].plot(t, s, lw=2, c='k')

    ax[0, 0].set_ylabel("v [mV]")
    ax[0, 1].set_ylabel('p')
    ax[1, 0].set_ylabel('q')
    ax[1, 1].set_ylabel('s')
    ax[1, 0].set_xlabel("time [ms]")
    ax[1, 1].set_xlabel("time [ms]")
    ax[0, 0].set_yticks(range(-100, 100, 50))
    ax[0, 0].set_ylim(-100, 50)
    ax[0, 1].set_xlim(min(t), max(t))
    ax[0, 1].set_ylim(0, 1.01)

    plt.tight_layout()
    return fig

In [ ]:
def _rtm_depressing_widget(U=0.5, tau_rec=500.):
    result = simulate_rtm_depressing(U=U, tau_rec=tau_rec)
    plot_rtm_depressing(result)


interact(_rtm_depressing_widget, U=(0.1, 0.9, 0.05), tau_rec=(50., 1000., 50.));

## WB with depressing synapse

`WB_WITH_DEPRESSING_S` gives the depressing-synapse case for a WB cell,
integrated with an explicit Heun (RK2) step rather than `odeint`. Compare
this to the RTM depression case above to separate synaptic from
intrinsic-neuron effects.

In [ ]:
# ------------------------------------------------------- WB gating functions

def alpha_m_wb(v):
    return 0.1 * (v + 35) / (1 - exp(-(v + 35) / 10))


def beta_m_wb(v):
    return 4. * exp(-(v + 60) / 18)


def m_inf_wb(v):
    return alpha_m_wb(v) / (alpha_m_wb(v) + beta_m_wb(v))


def alpha_h_wb(v):
    return 0.35 * exp(-(v + 58) / 20)


def beta_h_wb(v):
    return 5. / (exp(-0.1 * (v + 28)) + 1)


def h_inf_wb(v):
    return alpha_h_wb(v) / (alpha_h_wb(v) + beta_h_wb(v))


def alpha_n_wb(v):
    return 0.05 * (v + 34) / (1 - exp(-0.1 * (v + 34)))


def beta_n_wb(v):
    return 0.625 * exp(-(v + 44) / 80)


def n_inf_wb(v):
    return alpha_n_wb(v) / (alpha_n_wb(v) + beta_n_wb(v))

In [ ]:
def simulate_wb_with_depressing_s(i_ext=0.5, t_final=200., dt=0.01,
                                   U=0.5, C=1.25, tau_rec=500., tau_d_q=5.,
                                   tau_r=3., tau_d=9.):
    '''Note: unlike the WB neuron used elsewhere in this project (which
    divides tau_h/tau_n by a phi=5 speedup factor), this particular
    matlab source bakes phi=5 directly into the rate constants
    themselves (e.g. alpha_h=0.35*exp(...) = 5 * 0.07*exp(...)).
    alpha_m/beta_m are unaffected, as usual.'''
    c = 1.
    g_k_i, g_na_i, g_l_i = 9., 35., 0.1
    v_k_i, v_na_i, v_l_i = -90., 55., -65.

    dt05 = dt / 2
    m_steps = round(t_final / dt)

    v = np.zeros(m_steps + 1)
    m = np.zeros(m_steps + 1)
    h = np.zeros(m_steps + 1)
    n = np.zeros(m_steps + 1)
    p = np.zeros(m_steps + 1)
    q = np.zeros(m_steps + 1)
    s = np.zeros(m_steps + 1)

    v[0] = -70.
    m[0] = m_inf_wb(v[0])
    h[0] = h_inf_wb(v[0])
    n[0] = n_inf_wb(v[0])
    p[0] = 1.
    q[0] = 0.
    s[0] = 0.

    U_log = np.log(1 / (1 - U))
    num_spikes = 0

    for k in range(m_steps):
        v_inc = (g_k_i * n[k] ** 4 * (v_k_i - v[k]) + g_na_i * m[k] ** 3 * h[k] * (v_na_i - v[k])
                  + g_l_i * (v_l_i - v[k]) + i_ext) / c
        n_inc = alpha_n_wb(v[k]) * (1 - n[k]) - beta_n_wb(v[k]) * n[k]
        h_inc = alpha_h_wb(v[k]) * (1 - h[k]) - beta_h_wb(v[k]) * h[k]
        p_inc = -C * (1 + tanh(v[k] / 10)) * p[k] * U_log + (1 - p[k] - q[k]) / tau_rec
        q_inc = C * (1 + tanh(v[k] / 10)) * p[k] * U_log - q[k] / tau_d_q
        s_inc = q[k] * (1 - s[k]) / tau_r - s[k] / tau_d

        v_tmp = v[k] + dt05 * v_inc
        h_tmp = h[k] + dt05 * h_inc
        n_tmp = n[k] + dt05 * n_inc
        m_tmp = m_inf_wb(v_tmp)
        p_tmp = p[k] + dt05 * p_inc
        q_tmp = q[k] + dt05 * q_inc
        s_tmp = s[k] + dt05 * s_inc

        v_inc = (g_k_i * n_tmp ** 4 * (v_k_i - v_tmp) + g_na_i * m_tmp ** 3 * h_tmp * (v_na_i - v_tmp)
                  + g_l_i * (v_l_i - v_tmp) + i_ext) / c
        h_inc = alpha_h_wb(v_tmp) * (1 - h_tmp) - beta_h_wb(v_tmp) * h_tmp
        n_inc = alpha_n_wb(v_tmp) * (1 - n_tmp) - beta_n_wb(v_tmp) * n_tmp
        p_inc = -C * (1 + tanh(v_tmp / 10)) * p_tmp * U_log + (1 - p_tmp - q_tmp) / tau_rec
        q_inc = C * (1 + tanh(v_tmp / 10)) * p_tmp * U_log - q_tmp / tau_d_q
        s_inc = q_tmp * (1 - s_tmp) / tau_r - s_tmp / tau_d

        v[k + 1] = v[k] + dt * v_inc
        h[k + 1] = h[k] + dt * h_inc
        n[k + 1] = n[k] + dt * n_inc
        m[k + 1] = m_inf_wb(v[k + 1])
        p[k + 1] = p[k] + dt * p_inc
        q[k + 1] = q[k] + dt * q_inc
        s[k + 1] = s[k] + dt * s_inc

        if v[k + 1] < -20 and v[k] >= -20:
            num_spikes += 1

    gamma = C * (1 + tanh(v / 10))
    integral = gamma.sum() * dt / num_spikes if num_spikes else np.nan
    t = np.arange(m_steps + 1) * dt
    return SimpleNamespace(t=t, v=v, p=p, q=q, s=s, num_spikes=num_spikes, integral=integral)


def plot_wb_with_depressing_s(result):
    t, v, p, q, s = result.t, result.v, result.p, result.q, result.s
    fig, axes = plt.subplots(2, 2, figsize=(8, 6))
    axes[0, 0].plot(t, v, '-k', linewidth=2)
    axes[0, 0].set_ylabel('$v$ [mV]')
    axes[0, 0].axis([t[0], t[-1], -100, 50])

    axes[0, 1].plot(t, p, '-k', linewidth=2)
    axes[0, 1].set_ylabel('$p$')
    axes[0, 1].axis([t[0], t[-1], 0, 1])

    axes[1, 0].plot(t, q, '-k', linewidth=2)
    axes[1, 0].set_xlabel('$t$ [ms]')
    axes[1, 0].set_ylabel('$q$')
    axes[1, 0].axis([t[0], t[-1], 0, q.max() * 1.1])

    axes[1, 1].plot(t, s, '-k', linewidth=2)
    axes[1, 1].set_xlabel('$t$ [ms]')
    axes[1, 1].set_ylabel('$s$')
    axes[1, 1].axis([t[0], t[-1], 0, s.max() * 1.1])

    plt.tight_layout()
    return fig

In [ ]:
def _wb_with_depressing_s_widget(U=0.5, C=1.25):
    result = simulate_wb_with_depressing_s(U=U, C=C)
    print(f"integral per period: {result.integral:.4f}")
    plot_wb_with_depressing_s(result)


interact(_wb_with_depressing_s_widget, U=(0.1, 0.9, 0.05), C=(0.5, 3.0, 0.05));